# 10 — Statistical analysis and final figures

This notebook consumes only the frozen outputs produced by `09_final_model_family_benchmark.ipynb`. It does not fit, tune, or select any forecasting model.

The 40 forecasting tasks are treated as paired blocks. The analysis includes:

- Friedman comparison across the six model families;
- all pairwise Wilcoxon signed-rank contrasts with Holm correction;
- paired bootstrap confidence intervals for aggregated NRMSE differences;
- task-level Diebold–Mariano tests using a Newey–West long-run variance estimate;
- the final English task-winner and robust hybrid gain maps.

## Interpretation contract

NRMSE is the primary cross-task metric because temperature and relative humidity have different units. RMSE and $R^2$ remain the primary within-target descriptive metrics.

The main contrast is **Robust hybrid versus Advanced traditional**. Statistical non-rejection must not be rewritten as equivalence or superiority. The expected scientific interpretation is configuration-dependent performance: the hybrid correction can improve particular tasks, but it is not assumed to dominate globally.

In [ ]:
from __future__ import annotations

import itertools
import json
import os
import platform
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import seaborn as sns
from IPython.display import display
from scipy import stats
from statsmodels.stats.multitest import multipletests

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)
sns.set_theme(style="whitegrid", context="paper")

In [ ]:
def locate_project_root():
    explicit = os.environ.get("GREENHOUSE_PROJECT_ROOT")
    if explicit:
        root = Path(explicit).expanduser().resolve()
        if not (root / "results" / "final_benchmark").exists():
            raise FileNotFoundError(f"Notebook 09 outputs were not found under: {root}")
        return root
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "notebooks").exists() and (candidate / "results" / "final_benchmark").exists():
            return candidate
    raise FileNotFoundError(
        "Project root not found. Set GREENHOUSE_PROJECT_ROOT or run from the repository."
    )


PROJECT_ROOT = locate_project_root()
BENCHMARK_DIR = PROJECT_ROOT / "results" / "final_benchmark"
RESULTS_DIR = PROJECT_ROOT / "results" / "statistical_analysis"
FIGURE_DIR = PROJECT_ROOT / "figures" / "article"
METADATA_DIR = PROJECT_ROOT / "metadata"
for directory in [RESULTS_DIR, FIGURE_DIR, METADATA_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

EXECUTION_MODE = os.environ.get("GREENHOUSE_EXECUTION_MODE", "full").strip().lower()
if EXECUTION_MODE not in {"full", "smoke_test"}:
    raise ValueError("GREENHOUSE_EXECUTION_MODE must be 'full' or 'smoke_test'.")

TASK_COLUMNS = ["resolution_minutes", "target", "horizon_minutes"]
KEY_COLUMNS = TASK_COLUMNS + ["origin_index"]
FAMILIES = [
    "BASELINE", "STATISTICAL", "MACHINE_LEARNING",
    "DEEP_LEARNING", "ADVANCED_TRADITIONAL", "HYBRID_ROBUST",
]
FAMILY_LABELS = {
    "BASELINE": "Baseline",
    "STATISTICAL": "Statistical",
    "MACHINE_LEARNING": "Machine learning",
    "DEEP_LEARNING": "Deep learning",
    "ADVANCED_TRADITIONAL": "Advanced traditional",
    "HYBRID_ROBUST": "Robust hybrid",
}
FAMILY_ABBREVIATIONS = {
    "BASELINE": "B", "STATISTICAL": "S", "MACHINE_LEARNING": "ML",
    "DEEP_LEARNING": "DL", "ADVANCED_TRADITIONAL": "AT", "HYBRID_ROBUST": "RH",
}
FAMILY_COLORS = {
    "BASELINE": "#9E9E9E", "STATISTICAL": "#4C78A8",
    "MACHINE_LEARNING": "#F2CF5B", "DEEP_LEARNING": "#B279A2",
    "ADVANCED_TRADITIONAL": "#59A14F", "HYBRID_ROBUST": "#E15759",
    "SHARED": "#D9D9D9",
}
NUMERICAL_TIE_TOLERANCE = 1e-6
ALPHA = 0.05
BOOTSTRAP_REPETITIONS = 5000 if EXECUTION_MODE == "full" else 500
RANDOM_SEED = 2026

print({
    "project_root": str(PROJECT_ROOT),
    "execution_mode": EXECUTION_MODE,
    "bootstrap_repetitions": BOOTSTRAP_REPETITIONS,
})

## Load and validate notebook 09 outputs

In [ ]:
TASK_METRICS_FILE = BENCHMARK_DIR / "05_final_family_task_metrics.csv"
PREDICTIONS_FILE = BENCHMARK_DIR / "predictions" / "01_family_selected_test_predictions.parquet"
VALIDATION_FILE = BENCHMARK_DIR / "11_output_validation.csv"
for path in [TASK_METRICS_FILE, PREDICTIONS_FILE, VALIDATION_FILE]:
    if not path.exists():
        raise FileNotFoundError(f"Required notebook 09 output not found: {path}")

task_metrics = pd.read_csv(TASK_METRICS_FILE)
predictions = pd.read_parquet(PREDICTIONS_FILE)
upstream_validation = pd.read_csv(VALIDATION_FILE)
if not upstream_validation["passed"].astype(bool).all():
    raise AssertionError("Notebook 09 contains failed validation checks.")

required_metric_columns = set(["family", *TASK_COLUMNS, "rmse", "r2", "nrmse", "mae", "bias"])
required_prediction_columns = set(["family", *KEY_COLUMNS, "observed", "predicted"])
if missing := sorted(required_metric_columns - set(task_metrics.columns)):
    raise ValueError(f"Task metrics missing columns: {missing}")
if missing := sorted(required_prediction_columns - set(predictions.columns)):
    raise ValueError(f"Predictions missing columns: {missing}")

present_families = sorted(task_metrics["family"].unique())
assert set(present_families) == set(FAMILIES), present_families
task_counts = task_metrics.groupby("family")[TASK_COLUMNS].size()
assert task_counts.nunique() == 1, task_counts
N_TASKS = int(task_counts.iloc[0])
EXPECTED_TASKS = 40 if EXECUTION_MODE == "full" else 8
assert N_TASKS == EXPECTED_TASKS, {"observed": N_TASKS, "expected": EXPECTED_TASKS}

input_audit = pd.DataFrame([
    {"input": str(TASK_METRICS_FILE.relative_to(PROJECT_ROOT)), "rows": len(task_metrics), "columns": len(task_metrics.columns)},
    {"input": str(PREDICTIONS_FILE.relative_to(PROJECT_ROOT)), "rows": len(predictions), "columns": len(predictions.columns)},
    {"input": str(VALIDATION_FILE.relative_to(PROJECT_ROOT)), "rows": len(upstream_validation), "columns": len(upstream_validation.columns)},
])
input_audit.to_csv(RESULTS_DIR / "01_input_audit.csv", index=False)
display(input_audit)

## Friedman test across paired forecasting tasks

In [ ]:
nrmse_panel = task_metrics.pivot(index=TASK_COLUMNS, columns="family", values="nrmse")
nrmse_panel = nrmse_panel.reindex(columns=sorted(FAMILIES))
assert nrmse_panel.notna().all().all()

friedman_statistic, friedman_p_value = stats.friedmanchisquare(
    *[nrmse_panel[family].to_numpy(dtype=float) for family in nrmse_panel.columns]
)
friedman_result = pd.DataFrame([{
    "n_complete_tasks": len(nrmse_panel),
    "n_families": len(nrmse_panel.columns),
    "families": json.dumps(list(nrmse_panel.columns)),
    "metric": "NRMSE normalized by test-task target SD",
    "friedman_statistic": friedman_statistic,
    "p_value": friedman_p_value,
    "significant": friedman_p_value < ALPHA,
}])
friedman_result.to_csv(RESULTS_DIR / "02_friedman_family_comparison.csv", index=False)
display(friedman_result)

## Pairwise Wilcoxon signed-rank tests with Holm correction

In [ ]:
def paired_wilcoxon(family_a, family_b):
    difference = (
        nrmse_panel[family_a].to_numpy(dtype=float)
        - nrmse_panel[family_b].to_numpy(dtype=float)
    )
    numerical_ties = np.abs(difference) <= NUMERICAL_TIE_TOLERANCE
    adjusted_difference = difference.copy()
    adjusted_difference[numerical_ties] = 0.0
    nonzero = adjusted_difference[~numerical_ties]
    if len(nonzero) == 0:
        statistic, p_value = 0.0, 1.0
    else:
        result = stats.wilcoxon(nonzero, zero_method="wilcox", alternative="two-sided", method="auto")
        statistic, p_value = float(result.statistic), float(result.pvalue)
    favoring_a = int((adjusted_difference < 0).sum())
    favoring_b = int((adjusted_difference > 0).sum())
    n_nonzero = favoring_a + favoring_b
    rank_biserial_favoring_a = (favoring_a - favoring_b) / n_nonzero if n_nonzero else 0.0
    mean_difference = float(np.mean(adjusted_difference))
    if abs(mean_difference) <= NUMERICAL_TIE_TOLERANCE:
        direction = "NO_MEAN_DIFFERENCE"
    elif mean_difference < 0:
        direction = f"{family_a}_better"
    else:
        direction = f"{family_b}_better"
    return {
        "family_a": family_a,
        "family_b": family_b,
        "n_tasks": len(difference),
        "n_nonzero_tasks": n_nonzero,
        "n_numerical_ties": int(numerical_ties.sum()),
        "tasks_favoring_a": favoring_a,
        "tasks_favoring_b": favoring_b,
        "numerical_tie_tolerance": NUMERICAL_TIE_TOLERANCE,
        "median_nrmse_difference_a_minus_b": float(np.median(adjusted_difference)),
        "mean_nrmse_difference_a_minus_b": mean_difference,
        "rank_biserial_favoring_a": rank_biserial_favoring_a,
        "wilcoxon_statistic": statistic,
        "p_value": p_value,
        "direction": direction,
    }


wilcoxon_rows = [
    paired_wilcoxon(a, b)
    for a, b in itertools.combinations(sorted(FAMILIES), 2)
]
wilcoxon_results = pd.DataFrame(wilcoxon_rows)
wilcoxon_results["p_value_holm"] = multipletests(
    wilcoxon_results["p_value"], alpha=ALPHA, method="holm"
)[1]
wilcoxon_results["significant_after_holm"] = wilcoxon_results["p_value_holm"] < ALPHA
wilcoxon_results.to_csv(RESULTS_DIR / "03_pairwise_family_wilcoxon_holm.csv", index=False)
display(wilcoxon_results)

## Paired task bootstrap

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
bootstrap_rows = []
for family_a, family_b in itertools.combinations(sorted(FAMILIES), 2):
    difference = (
        nrmse_panel[family_a].to_numpy(dtype=float)
        - nrmse_panel[family_b].to_numpy(dtype=float)
    )
    difference[np.abs(difference) <= NUMERICAL_TIE_TOLERANCE] = 0.0
    sample_indices = rng.integers(0, len(difference), size=(BOOTSTRAP_REPETITIONS, len(difference)))
    samples = difference[sample_indices]
    boot_mean = samples.mean(axis=1)
    boot_median = np.median(samples, axis=1)
    mean_ci = np.quantile(boot_mean, [0.025, 0.975])
    median_ci = np.quantile(boot_median, [0.025, 0.975])
    bootstrap_rows.append({
        "family_a": family_a, "family_b": family_b,
        "metric": "NRMSE normalized by test-task target SD",
        "n_tasks": len(difference), "bootstrap_repetitions": BOOTSTRAP_REPETITIONS,
        "mean_difference": float(difference.mean()),
        "median_difference": float(np.median(difference)),
        "mean_ci_lower": float(mean_ci[0]), "mean_ci_upper": float(mean_ci[1]),
        "median_ci_lower": float(median_ci[0]), "median_ci_upper": float(median_ci[1]),
        "mean_ci_excludes_zero": bool(mean_ci[0] > 0 or mean_ci[1] < 0),
        "median_ci_excludes_zero": bool(median_ci[0] > 0 or median_ci[1] < 0),
    })

bootstrap_results = pd.DataFrame(bootstrap_rows)
bootstrap_results.to_csv(RESULTS_DIR / "04_family_paired_bootstrap.csv", index=False)
display(bootstrap_results)

## Diebold–Mariano tests with Newey–West variance

For each task and each contrast involving the robust hybrid strategy, the loss differential is squared error of `HYBRID_ROBUST` minus squared error of the comparison family. A negative mean differential favors the hybrid strategy. The Bartlett-weighted Newey–West lag is the forecast horizon in resolution steps minus one, capped at $n-1$. Holm correction is applied globally across all finite task-level tests.

In [ ]:
def newey_west_long_run_variance(values, maximum_lag):
    values = np.asarray(values, dtype=float)
    centered = values - values.mean()
    n = len(centered)
    maximum_lag = int(min(maximum_lag, n - 1))
    variance = float(np.dot(centered, centered) / n)
    for lag in range(1, maximum_lag + 1):
        weight = 1.0 - lag / (maximum_lag + 1.0)
        covariance = float(np.dot(centered[lag:], centered[:-lag]) / n)
        variance += 2.0 * weight * covariance
    return max(variance, 0.0)


def dm_test(loss_differential, maximum_lag):
    difference = np.asarray(loss_differential, dtype=float)
    if np.all(np.abs(difference) <= 1e-14):
        return 0.0, 1.0, 0.0, 0.0, "exact_loss_tie"
    long_run_variance = newey_west_long_run_variance(difference, maximum_lag)
    standard_error = np.sqrt(long_run_variance / len(difference))
    if not np.isfinite(standard_error) or standard_error <= 0:
        return np.nan, np.nan, float(difference.mean()), long_run_variance, "nonpositive_long_run_variance"
    statistic = float(difference.mean() / standard_error)
    p_value = float(2.0 * stats.norm.sf(abs(statistic)))
    return statistic, p_value, float(difference.mean()), long_run_variance, "ok"


wide_predictions = predictions.pivot(
    index=KEY_COLUMNS + ["observed"], columns="family", values="predicted"
).reset_index()
dm_rows = []
for comparison_family in [family for family in FAMILIES if family != "HYBRID_ROBUST"]:
    for task_keys, group in wide_predictions.groupby(TASK_COLUMNS, sort=False, observed=True):
        observed = group["observed"].to_numpy(dtype=float)
        hybrid_error = group["HYBRID_ROBUST"].to_numpy(dtype=float) - observed
        comparison_error = group[comparison_family].to_numpy(dtype=float) - observed
        loss_difference = hybrid_error ** 2 - comparison_error ** 2
        resolution, target, horizon = task_keys
        horizon_steps = max(1, int(round(horizon / resolution)))
        maximum_lag = min(horizon_steps - 1, len(group) - 1)
        statistic, p_value, mean_difference, long_run_variance, status = dm_test(
            loss_difference, maximum_lag
        )
        dm_rows.append({
            "family_a": "HYBRID_ROBUST", "family_b": comparison_family,
            "resolution_minutes": int(resolution), "target": target,
            "horizon_minutes": int(horizon), "n_origins": len(group),
            "newey_west_lag": maximum_lag,
            "mean_squared_error_difference_a_minus_b": mean_difference,
            "newey_west_long_run_variance": long_run_variance,
            "dm_statistic": statistic, "p_value": p_value, "status": status,
            "direction": (
                "HYBRID_ROBUST_better" if mean_difference < -1e-14
                else f"{comparison_family}_better" if mean_difference > 1e-14
                else "EXACT_TIE"
            ),
        })

dm_results = pd.DataFrame(dm_rows)
finite = dm_results["p_value"].notna()
dm_results["p_value_holm_global"] = np.nan
if finite.any():
    dm_results.loc[finite, "p_value_holm_global"] = multipletests(
        dm_results.loc[finite, "p_value"], alpha=ALPHA, method="holm"
    )[1]
dm_results["significant_after_global_holm"] = dm_results["p_value_holm_global"] < ALPHA
dm_results.to_csv(RESULTS_DIR / "05_dm_hybrid_primary_contrasts.csv", index=False)

dm_summary = (
    dm_results.groupby(["family_a", "family_b"], as_index=False)
    .agg(
        tasks=("p_value", "size"), exact_ties=("status", lambda s: int((s == "exact_loss_tie").sum())),
        hybrid_better_tasks=("direction", lambda s: int((s == "HYBRID_ROBUST_better").sum())),
        comparison_better_tasks=("direction", lambda s: int(s.str.endswith("_better").sum() - (s == "HYBRID_ROBUST_better").sum())),
        significant_tasks_global_holm=("significant_after_global_holm", "sum"),
    )
)
dm_summary.to_csv(RESULTS_DIR / "06_dm_hybrid_primary_summary.csv", index=False)
display(dm_summary)

## Descriptive task winners and the primary hybrid contrast

In [ ]:
winner_rows = []
for task_keys, group in task_metrics.groupby(TASK_COLUMNS, sort=False, observed=True):
    minimum_rmse = float(group["rmse"].min())
    winners = sorted(group.loc[np.abs(group["rmse"] - minimum_rmse) <= NUMERICAL_TIE_TOLERANCE, "family"])
    winner_rows.append({
        "resolution_minutes": int(task_keys[0]), "target": task_keys[1],
        "horizon_minutes": int(task_keys[2]), "minimum_rmse": minimum_rmse,
        "winning_families": ";".join(winners), "n_winners": len(winners),
        "winner_label": " / ".join(FAMILY_ABBREVIATIONS[x] for x in winners),
        "winner_category": winners[0] if len(winners) == 1 else "SHARED",
    })
task_winners = pd.DataFrame(winner_rows)
task_winners.to_csv(RESULTS_DIR / "07_task_winners_rmse.csv", index=False)

hybrid_comparison = task_metrics.loc[
    task_metrics["family"].isin(["ADVANCED_TRADITIONAL", "HYBRID_ROBUST"])
].pivot(index=TASK_COLUMNS, columns="family", values="rmse").reset_index()
hybrid_comparison["hybrid_gain_pct"] = 100.0 * (
    hybrid_comparison["ADVANCED_TRADITIONAL"] - hybrid_comparison["HYBRID_ROBUST"]
) / hybrid_comparison["ADVANCED_TRADITIONAL"]
hybrid_comparison["outcome"] = np.select(
    [
        np.abs(hybrid_comparison["HYBRID_ROBUST"] - hybrid_comparison["ADVANCED_TRADITIONAL"]) <= NUMERICAL_TIE_TOLERANCE,
        hybrid_comparison["HYBRID_ROBUST"] < hybrid_comparison["ADVANCED_TRADITIONAL"],
    ],
    ["NUMERICAL_TIE", "HYBRID_ROBUST_BETTER"],
    default="ADVANCED_TRADITIONAL_BETTER",
)
hybrid_comparison.to_csv(RESULTS_DIR / "08_hybrid_vs_advanced_task_comparison.csv", index=False)
hybrid_summary = hybrid_comparison["outcome"].value_counts().rename_axis("outcome").reset_index(name="tasks")
hybrid_summary.to_csv(RESULTS_DIR / "09_hybrid_vs_advanced_summary.csv", index=False)
display(hybrid_summary)

## Figure 4 — Task-winning model family

In [ ]:
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

target_order = ["temperature", "relative_humidity"]
target_labels = {"temperature": "Air temperature", "relative_humidity": "Relative humidity"}
resolutions = sorted(task_winners["resolution_minutes"].unique())
horizons = sorted(task_winners["horizon_minutes"].unique())
categories = [*FAMILIES, "SHARED"]
category_to_number = {category: index for index, category in enumerate(categories)}
cmap = ListedColormap([FAMILY_COLORS[category] for category in categories])

fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.8), constrained_layout=True)
for axis, target in zip(axes, target_order):
    subset = task_winners.loc[task_winners["target"].eq(target)]
    category_matrix = subset.pivot(
        index="resolution_minutes", columns="horizon_minutes", values="winner_category"
    ).reindex(index=resolutions, columns=horizons)
    label_matrix = subset.pivot(
        index="resolution_minutes", columns="horizon_minutes", values="winner_label"
    ).reindex(index=resolutions, columns=horizons)
    numeric = category_matrix.map(category_to_number.get).to_numpy(dtype=float)
    axis.imshow(numeric, cmap=cmap, vmin=-0.5, vmax=len(categories)-0.5, aspect="auto")
    for row in range(len(resolutions)):
        for column in range(len(horizons)):
            axis.text(column, row, label_matrix.iloc[row, column], ha="center", va="center", fontsize=9, fontweight="bold")
    axis.set_xticks(range(len(horizons)), [str(x) for x in horizons])
    axis.set_yticks(range(len(resolutions)), [str(x) for x in resolutions])
    axis.set_xlabel("Forecast horizon (min)")
    axis.set_ylabel("Temporal resolution (min)")
    axis.set_title(target_labels[target])
    axis.set_xticks(np.arange(-0.5, len(horizons), 1), minor=True)
    axis.set_yticks(np.arange(-0.5, len(resolutions), 1), minor=True)
    axis.grid(which="minor", color="white", linewidth=1.5)
    axis.tick_params(which="minor", bottom=False, left=False)

used_categories = [category for category in categories if category in set(task_winners["winner_category"])]
legend = [
    Patch(facecolor=FAMILY_COLORS[category], label=("Shared minimum" if category == "SHARED" else FAMILY_LABELS[category]))
    for category in used_categories
]
fig.legend(handles=legend, loc="outside lower center", ncol=min(4, len(legend)), frameon=False)
fig.suptitle("Best-performing model family by forecasting task (RMSE)", fontsize=12, fontweight="bold")
for extension in ["png", "pdf"]:
    fig.savefig(FIGURE_DIR / f"04_family_winners_rmse.{extension}", dpi=300, bbox_inches="tight")
plt.show()

## Figure 5 — Robust hybrid RMSE gain relative to Advanced traditional

In [ ]:
gain_limit = max(1.0, float(np.nanmax(np.abs(hybrid_comparison["hybrid_gain_pct"]))))
fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.5), constrained_layout=True)
for axis, target in zip(axes, target_order):
    matrix = hybrid_comparison.loc[hybrid_comparison["target"].eq(target)].pivot(
        index="resolution_minutes", columns="horizon_minutes", values="hybrid_gain_pct"
    ).reindex(index=resolutions, columns=horizons)
    sns.heatmap(
        matrix, ax=axis, cmap="RdBu_r", center=0, vmin=-gain_limit, vmax=gain_limit,
        annot=True, fmt=".2f", linewidths=0.75, linecolor="white",
        cbar=axis is axes[-1],
        cbar_kws={"label": "RMSE reduction (%)"} if axis is axes[-1] else None,
    )
    axis.set_xlabel("Forecast horizon (min)")
    axis.set_ylabel("Temporal resolution (min)")
    axis.set_title(target_labels[target])
    axis.set_yticklabels(axis.get_yticklabels(), rotation=0)

fig.suptitle(
    "Robust hybrid RMSE change relative to Advanced traditional\nPositive values indicate lower RMSE for Robust hybrid",
    fontsize=12, fontweight="bold",
)
for extension in ["png", "pdf"]:
    fig.savefig(FIGURE_DIR / f"05_hybrid_vs_advanced_rmse_gain.{extension}", dpi=300, bbox_inches="tight")
plt.show()

## Supplementary average-rank figure

In [ ]:
average_ranks = (
    task_metrics.assign(
        rmse_rank=lambda x: x.groupby(TASK_COLUMNS)["rmse"].rank(method="average")
    ).groupby("family", as_index=False)
    .agg(average_rmse_rank=("rmse_rank", "mean"), median_rmse_rank=("rmse_rank", "median"))
    .sort_values("average_rmse_rank")
)
average_ranks["family_label"] = average_ranks["family"].map(FAMILY_LABELS)
average_ranks.to_csv(RESULTS_DIR / "10_average_family_ranks.csv", index=False)

fig, axis = plt.subplots(figsize=(7.5, 4.2), constrained_layout=True)
axis.barh(
    average_ranks["family_label"], average_ranks["average_rmse_rank"],
    color=[FAMILY_COLORS[x] for x in average_ranks["family"]],
)
axis.invert_yaxis()
axis.set_xlabel("Average RMSE rank (lower is better)")
axis.set_ylabel("")
axis.set_title("Average family rank across paired forecasting tasks", fontweight="bold")
axis.set_xlim(0.5, len(FAMILIES) + 0.2)
for row, value in enumerate(average_ranks["average_rmse_rank"]):
    axis.text(value + 0.05, row, f"{value:.2f}", va="center")
for extension in ["png", "pdf"]:
    fig.savefig(FIGURE_DIR / f"S01_average_family_rank.{extension}", dpi=300, bbox_inches="tight")
plt.show()

## Historical audit and manuscript-ready statistical summary

In [ ]:
primary_wilcoxon = wilcoxon_results.loc[
    (wilcoxon_results["family_a"].eq("ADVANCED_TRADITIONAL") & wilcoxon_results["family_b"].eq("HYBRID_ROBUST"))
    | (wilcoxon_results["family_a"].eq("HYBRID_ROBUST") & wilcoxon_results["family_b"].eq("ADVANCED_TRADITIONAL"))
].iloc[0]
primary_bootstrap = bootstrap_results.loc[
    (bootstrap_results["family_a"].eq("ADVANCED_TRADITIONAL") & bootstrap_results["family_b"].eq("HYBRID_ROBUST"))
    | (bootstrap_results["family_a"].eq("HYBRID_ROBUST") & bootstrap_results["family_b"].eq("ADVANCED_TRADITIONAL"))
].iloc[0]

historical_audit = pd.DataFrame([
    {"quantity": "Friedman statistic", "observed": friedman_statistic, "historical": 86.90962099125365, "absolute_difference": abs(friedman_statistic - 86.90962099125365)},
    {"quantity": "Friedman p-value", "observed": friedman_p_value, "historical": 2.993224842379209e-17, "absolute_difference": abs(friedman_p_value - 2.993224842379209e-17)},
    {"quantity": "Advanced vs hybrid Holm p-value", "observed": primary_wilcoxon.p_value_holm, "historical": 0.208984375, "absolute_difference": abs(primary_wilcoxon.p_value_holm - 0.208984375)},
    {"quantity": "Advanced minus hybrid mean NRMSE", "observed": primary_bootstrap.mean_difference, "historical": 0.0004047030991098571, "absolute_difference": abs(primary_bootstrap.mean_difference - 0.0004047030991098571)},
])
historical_audit["within_0_001"] = historical_audit["absolute_difference"] <= 0.001
historical_audit.to_csv(RESULTS_DIR / "11_historical_statistical_audit.csv", index=False)

if primary_wilcoxon.p_value_holm < ALPHA:
    primary_interpretation = "The paired Holm-adjusted contrast detected a difference between the robust hybrid and advanced traditional families."
else:
    primary_interpretation = "The paired Holm-adjusted contrast did not detect an overall difference between the robust hybrid and advanced traditional families."

summary = {
    "friedman_statistic": float(friedman_statistic),
    "friedman_p_value": float(friedman_p_value),
    "primary_contrast_holm_p_value": float(primary_wilcoxon.p_value_holm),
    "primary_bootstrap_mean_difference": float(primary_bootstrap.mean_difference),
    "primary_bootstrap_mean_ci": [float(primary_bootstrap.mean_ci_lower), float(primary_bootstrap.mean_ci_upper)],
    "primary_interpretation": primary_interpretation,
    "required_conclusion": "Performance differences are task-dependent; no universal robust hybrid superiority is claimed.",
}
(RESULTS_DIR / "12_manuscript_statistical_summary.json").write_text(
    json.dumps(summary, indent=2), encoding="utf-8"
)
display(historical_audit)
print(json.dumps(summary, indent=2))

## Final validation and runtime metadata

In [ ]:
expected_dm_rows = (len(FAMILIES) - 1) * N_TASKS
validation_report = pd.DataFrame([
    {"check": "complete_nrmse_panel", "passed": bool(nrmse_panel.notna().all().all()), "detail": f"tasks={len(nrmse_panel)}, families={len(nrmse_panel.columns)}"},
    {"check": "friedman_result_finite", "passed": bool(np.isfinite(friedman_statistic) and np.isfinite(friedman_p_value)), "detail": f"statistic={friedman_statistic}, p={friedman_p_value}"},
    {"check": "all_pairwise_contrasts", "passed": len(wilcoxon_results) == 15, "detail": f"rows={len(wilcoxon_results)}"},
    {"check": "bootstrap_complete", "passed": len(bootstrap_results) == 15, "detail": f"rows={len(bootstrap_results)}, repetitions={BOOTSTRAP_REPETITIONS}"},
    {"check": "dm_primary_contrasts_complete", "passed": len(dm_results) == expected_dm_rows, "detail": f"rows={len(dm_results)}, expected={expected_dm_rows}"},
    {"check": "task_winner_map_complete", "passed": len(task_winners) == N_TASKS, "detail": f"rows={len(task_winners)}"},
    {"check": "hybrid_comparison_complete", "passed": len(hybrid_comparison) == N_TASKS, "detail": f"rows={len(hybrid_comparison)}"},
    {"check": "figures_created", "passed": all((FIGURE_DIR / name).exists() for name in ["04_family_winners_rmse.png", "05_hybrid_vs_advanced_rmse_gain.png", "S01_average_family_rank.png"]), "detail": str(FIGURE_DIR.relative_to(PROJECT_ROOT))},
])
assert validation_report["passed"].all(), validation_report
validation_report.to_csv(RESULTS_DIR / "13_output_validation.csv", index=False)

configuration = {
    "notebook": "10_statistical_analysis_and_figures.ipynb",
    "execution_mode": EXECUTION_MODE,
    "paired_blocks": N_TASKS,
    "primary_metric": "NRMSE normalized by test-task target SD",
    "omnibus_test": "Friedman",
    "post_hoc_test": "paired Wilcoxon signed-rank",
    "multiplicity_correction": "Holm",
    "numerical_tie_tolerance": NUMERICAL_TIE_TOLERANCE,
    "bootstrap_repetitions": BOOTSTRAP_REPETITIONS,
    "bootstrap_seed": RANDOM_SEED,
    "dm_loss": "squared error",
    "dm_variance": "Newey-West long-run variance with Bartlett weights",
    "dm_holm_scope": "all robust hybrid task-level contrasts",
    "primary_contrast": ["HYBRID_ROBUST", "ADVANCED_TRADITIONAL"],
}
(METADATA_DIR / "10_statistical_analysis_configuration.json").write_text(
    json.dumps(configuration, indent=2), encoding="utf-8"
)
runtime = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "python": platform.python_version(), "numpy": np.__version__,
    "pandas": pd.__version__, "scipy": scipy.__version__,
}
(METADATA_DIR / "10_statistical_analysis_runtime.json").write_text(
    json.dumps(runtime, indent=2), encoding="utf-8"
)
display(validation_report)

## Completion criteria

A full run must produce:

- one Friedman result across 40 paired tasks and six families;
- 15 Wilcoxon contrasts with global Holm correction;
- 15 paired-bootstrap contrasts using 5,000 resamples;
- 200 task-level Diebold–Mariano results for Robust hybrid against the other five families;
- English Figure 4 and Figure 5 in PNG and PDF formats;
- a manuscript-ready statistical summary that does not overstate the robust hybrid evidence.

Historical results are used only as a reproducibility audit. They are never copied into the new outputs.